# S08 - RNN & LSTM
## Exercises

### Exercise 1 (Easy)
Create a simple RNN cell and pass a sequence through it.

In [2]:
import torch
import torch.nn as nn

# Create RNN: input_size=10, hidden_size=20
cell = nn.RNNCell(input_size=10, hidden_size=20)
# Pass a sequence of shape (seq_len=5, batch=1, input_size=10)
seq_len, batch, input_size = 5, 1, 10

x = torch.randn(seq_len, batch, input_size)

h_t = torch.zeros(batch, 20)

for t in range(seq_len):
    h_t = cell(x[t], h_t)
    print(f"Step {t}, h_t: {h_t}")

Step 0, h_t: tensor([[ 0.3126, -0.4639,  0.3548,  0.2998,  0.1433, -0.1974,  0.1589, -0.2031,
         -0.5620,  0.1450, -0.5277,  0.1012,  0.6123, -0.1436,  0.1717, -0.2288,
          0.0607,  0.2613, -0.2530, -0.1201]], grad_fn=<TanhBackward0>)
Step 1, h_t: tensor([[-0.3147, -0.1766, -0.1239,  0.0733,  0.6528, -0.5743, -0.1436,  0.0909,
          0.0471, -0.2187, -0.1165, -0.0899, -0.3318,  0.0345, -0.0892,  0.4358,
         -0.1562,  0.6107,  0.2977,  0.2123]], grad_fn=<TanhBackward0>)
Step 2, h_t: tensor([[ 0.1841,  0.1899,  0.1852,  0.4239,  0.4247, -0.3712,  0.3377,  0.3049,
         -0.2682, -0.1714,  0.4005, -0.6101,  0.0501,  0.5506,  0.3910, -0.0721,
          0.4778,  0.4866,  0.7509,  0.3648]], grad_fn=<TanhBackward0>)
Step 3, h_t: tensor([[-0.2345, -0.4708,  0.1734,  0.1947, -0.2490, -0.2319, -0.0801, -0.0868,
         -0.4715,  0.5301, -0.5604,  0.6127,  0.5815, -0.5193, -0.2832, -0.4199,
         -0.2288,  0.1380, -0.1702, -0.1621]], grad_fn=<TanhBackward0>)
Step 4, h_t:

### Exercise 2 (Easy)
Create an LSTM and compare output shapes with vanilla RNN.

In [3]:
# Create LSTM with same dimensions
# Note: LSTM returns (output, (h_n, c_n))
seq_len, batch, input_size = 5, 1, 10

lstm = nn.LSTMCell(input_size, hidden_size=20)

x = torch.randn(seq_len, batch, input_size)

h_t = torch.zeros(batch, 20)
c_t = torch.zeros(batch, 20)

for t in range(seq_len):
    h_t, c_t = lstm(x[t], (h_t, c_t))
    print(f"Step {t}\n h_t: {h_t}\n c_t: {c_t}")

Step 0
 h_t: tensor([[-0.1378, -0.0147, -0.0147,  0.1225,  0.1930, -0.0011,  0.0508,  0.1020,
          0.0249,  0.0185, -0.1988,  0.1027,  0.0129, -0.1425, -0.1339, -0.1064,
         -0.1473,  0.1975,  0.0415,  0.0370]], grad_fn=<MulBackward0>)
 c_t: tensor([[-0.2793, -0.0259, -0.0472,  0.3692,  0.2519, -0.0036,  0.1082,  0.2488,
          0.0847,  0.0310, -0.3286,  0.1681,  0.0278, -0.1885, -0.2277, -0.1788,
         -0.3111,  0.4511,  0.0698,  0.0811]], grad_fn=<AddBackward0>)
Step 1
 h_t: tensor([[-0.2348,  0.0677,  0.0072,  0.1379,  0.1567,  0.0040,  0.0433,  0.1092,
          0.0818, -0.0178, -0.2323,  0.0549,  0.0206, -0.0563, -0.1157, -0.1413,
         -0.1047,  0.2450,  0.0083,  0.0431]], grad_fn=<MulBackward0>)
 c_t: tensor([[-0.4557,  0.1354,  0.0190,  0.3159,  0.2214,  0.0090,  0.0780,  0.2271,
          0.2295, -0.0401, -0.5395,  0.1060,  0.0330, -0.0873, -0.2701, -0.2790,
         -0.1642,  0.5704,  0.0175,  0.0723]], grad_fn=<AddBackward0>)
Step 2
 h_t: tensor([[-0.0913,

### Exercise 3 (Medium)
Build an LSTM-based sentiment classifier.

In [6]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        # Define: embedding, lstm, fc
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(in_features=hidden_dim, out_features=num_classes)
    
    def forward(self, x):
        # Embed -> LSTM -> take last hidden -> classify
        embeddings = self.embedding(x)

        output, (h_n, _) = self.lstm(embeddings)

        last_hidden = h_n[-1]

        return self.fc(last_hidden)

# Test with dummy data
# Parameters
vocab_size = 50
embed_dim = 16
hidden_dim = 32
num_classes = 5
batch_size = 4
seq_len = 10

x = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long)

model = LSTMClassifier(vocab_size, embed_dim, hidden_dim, num_classes)
logits = model.forward(x)

print(logits)

tensor([[ 0.0410, -0.0115, -0.1168, -0.1512, -0.2153],
        [-0.0034, -0.0600, -0.0419, -0.0226, -0.1641],
        [ 0.0811, -0.0035, -0.0496, -0.0717, -0.1000],
        [ 0.0123, -0.0242, -0.0885,  0.0685, -0.1471]],
       grad_fn=<AddmmBackward0>)


### Exercise 4 (Medium)
Implement a character-level language model with LSTM.

In [8]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # Embedding -> LSTM -> Linear (predict next char)
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(in_features=hidden_dim, out_features=vocab_size)
    
    def forward(self, x, hidden=None):
        x = self.embedding(x)
        x, hidden = self.lstm(x, hidden)
        x = self.fc(x)
        return x, hidden

# Train on a simple text and generate characters
text = "hello world this is a training example for char lstm"
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}
embed_dim = 16
hidden_dim = 32
model = CharLSTM(vocab_size, embed_dim, hidden_dim)
# Prepare training data
seq_len = 10
inputs = []
targets = []
for i in range(len(text) - seq_len):
    input_seq = text[i:i+seq_len]
    target_char = text[i+seq_len]
    inputs.append([char_to_idx[ch] for ch in input_seq])
    targets.append(char_to_idx[target_char])
inputs = torch.tensor(inputs, dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)
# Train the model
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs, _ = model(inputs)
    last_output = outputs[:, -1, :]  # [batch, vocab_size]
    loss = criterion(last_output, targets)  # targets: [batch]
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")
# Generate text
model.eval()
start_seq = "hello worl"
input_seq = torch.tensor([[char_to_idx[ch] for ch in start_seq]], dtype=torch.long)
hidden = None
generated_text = start_seq
for _ in range(50):
    output, hidden = model(input_seq, hidden)
    next_char_idx = output[:, -1, :].argmax(dim=-1).item()
    next_char = idx_to_char[next_char_idx]
    generated_text += next_char
    input_seq = torch.tensor([[next_char_idx]], dtype=torch.long)
print("Generated text:")
print(generated_text)

Epoch 20/200, Loss: 1.1638
Epoch 40/200, Loss: 0.1115
Epoch 60/200, Loss: 0.0208
Epoch 80/200, Loss: 0.0101
Epoch 100/200, Loss: 0.0068
Epoch 120/200, Loss: 0.0051
Epoch 140/200, Loss: 0.0040
Epoch 160/200, Loss: 0.0033
Epoch 180/200, Loss: 0.0027
Epoch 200/200, Loss: 0.0023
Generated text:
hello world thisa ing e examplexample for char lstmis a is t


### Exercise 5 (Hard)
Implement a bidirectional LSTM for sequence labeling (e.g., POS tagging).

*Research: BiLSTM processes sequence in both directions and concatenates outputs.*

In [ ]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags):
        super().__init__()
        # BiLSTM: hidden_dim * 2 for output
        pass
    
    def forward(self, x):
        # Return tag scores for each position
        pass
